# Two slides for a manager — reproducible calculations

Two questions, one slide each. This notebook reproduces every number on the slides.

- **Slide 1** — How many accelerators do OpenAI and Anthropic operate, and how is the fleet split between training and inference (May 2026)?
- **Slide 2** — Does Atlas Cloud make money reselling DeepSeek V4 Pro at \$1.68 / 1M input and \$3.38 / 1M output tokens?

All assumptions are sourced in [`sources.md`](sources.md). Pure standard library — runs with any Python 3.8+ (`python3 model.py` gives the same output). Code & comments in English; slides in Russian.

> Method note: the honest answer to Slide 1 separates **operational** fleet (running today) from **contracted/announced** capacity (tens of GW for 2026–2029). We report ranges (low/base/high), not false-precision point estimates.

> **v2** adds an explicit **bottom-up TCO** section for Slide 2: the node cost is rebuilt from first principles (hardware amortization + power + datacenter + staff) and compared with the rental-rate proxy.

In [1]:
from dataclasses import dataclass

def fmt_m(x):
    return f"{x/1e6:.2f}M"

def pct(x):
    return f"{x*100:.0f}%"

## Slide 1 — Fleet size & training/inference split

Fleets are normalized to **H100-equivalents** because Anthropic runs mostly non-NVIDIA ASICs (AWS Trainium, Google TPU), so raw chip counts are not comparable.

**Conversion factor** (sanity check): ~1.5 kW per accelerator all-in (chip TDP + rack + PUE 1.1–1.2) ⇒ ~667k H100-class chips per GW.

**Anchors.** OpenAI: Epoch AI estimated ~1.1M H100-eq mid-2025 (CI 0.8–1.4M); +~3 quarters of Blackwell ramp → ~2.0M base. Anthropic: bottom-up across Trainium2 (~0.5–1.0M units ×0.7), Google TPU (~0.2–1.0M mixed-gen ×~1.2) and NVIDIA (~0.25M incl. a temporary SpaceX lease) → ~1.7M base.

In [2]:
KW_PER_CHIP_ALLIN = 1.5                 # facility wall power per accelerator (1.3-1.7)
CHIPS_PER_GW = 1e6 / KW_PER_CHIP_ALLIN  # ~667k H100-class chips per GW
EQ_PER_PHYSICAL_CHIP = 1.3              # blended H100-eq per physical accelerator (GB200 ~3x/chip)

@dataclass
class Fleet:
    name: str
    h100eq_low: float; h100eq_base: float; h100eq_high: float
    inference_share: float; inference_share_lo: float; inference_share_hi: float
    note: str

# OpenAI base: Hopper legacy ~1.25M H100-eq + ~0.25M GB200 (x3 -> ~0.75M) -> ~2.0M.
# Anthropic base: Trainium2 ~0.8M (x0.7) + TPU ~0.6M (x~1.2) + NVIDIA ~0.25M incl. SpaceX Colossus (x~1.6) -> ~1.7M.
openai = Fleet('OpenAI', 1.4e6, 2.0e6, 3.0e6, 0.48, 0.45, 0.55,
               'Almost entirely NVIDIA (Hopper legacy + GB200 Blackwell).')
anthropic = Fleet('Anthropic', 1.0e6, 1.7e6, 3.0e6, 0.65, 0.55, 0.75,
                  'Mostly ASICs: AWS Trainium (training) + Google TPU (inference); little NVIDIA.')

INDUSTRY_INFERENCE_SHARE = 0.65         # Barclays >70% by 2026; Jensen: "vast majority is inference"; McKinsey >half of DC spend

CONTRACTED = {
    'OpenAI':    "~30+ GW announced (NVIDIA 10 + AMD 6 + Broadcom 10 + Oracle 4.5); '4-5M GPUs' by 2029.",
    'Anthropic': 'up to 5 GW AWS Trainium + ~1M Google TPU (1 GW, 2026) + ~3.5 GW Google/Broadcom (2027+) + up to 1 GW Azure.',
}

for f in (openai, anthropic):
    inf  = f.h100eq_base * f.inference_share
    tr   = f.h100eq_base * (1 - f.inference_share)
    print(f"{f.name}")
    print(f"  Fleet (H100-eq): low {fmt_m(f.h100eq_low)} | base {fmt_m(f.h100eq_base)} | high {fmt_m(f.h100eq_high)}")
    print(f"  Split (base): inference {pct(f.inference_share)} (~{fmt_m(inf)}) | training+R&D {pct(1-f.inference_share)} (~{fmt_m(tr)})")
    print(f"  Split range:  inference {pct(f.inference_share_lo)}-{pct(f.inference_share_hi)}")
    print(f"  {f.note}")
    print(f"  CONTRACTED (not today): {CONTRACTED[f.name]}\n")

print(f"Industry reference: ~{pct(INDUSTRY_INFERENCE_SHARE)} of AI compute is now inference.")
oa_chips = openai.h100eq_base / EQ_PER_PHYSICAL_CHIP
print(f"Sanity check: 1 GW ~= {CHIPS_PER_GW:,.0f} H100-class chips. OpenAI's {fmt_m(openai.h100eq_base)} H100-eq")
print(f"  ~= {fmt_m(oa_chips)} physical accelerators (GB200 ~3x/chip) ~= {oa_chips/CHIPS_PER_GW:.1f} GW across ALL clouds")
print(f"  (Azure + Oracle + CoreWeave + Stargate); Stargate/Abilene alone ~0.3 GW live, ~1.2 GW projected Q4-2026.")


OpenAI
  Fleet (H100-eq): low 1.40M | base 2.00M | high 3.00M
  Split (base): inference 48% (~0.96M) | training+R&D 52% (~1.04M)
  Split range:  inference 45%-55%
  Almost entirely NVIDIA (Hopper legacy + GB200 Blackwell).
  CONTRACTED (not today): ~30+ GW announced (NVIDIA 10 + AMD 6 + Broadcom 10 + Oracle 4.5); '4-5M GPUs' by 2029.

Anthropic
  Fleet (H100-eq): low 1.00M | base 1.70M | high 3.00M
  Split (base): inference 65% (~1.10M) | training+R&D 35% (~0.59M)
  Split range:  inference 55%-75%
  Mostly ASICs: AWS Trainium (training) + Google TPU (inference); little NVIDIA.
  CONTRACTED (not today): up to 5 GW AWS Trainium + ~1M Google TPU (1 GW, 2026) + ~3.5 GW Google/Broadcom (2027+) + up to 1 GW Azure.

Industry reference: ~65% of AI compute is now inference.
Sanity check: 1 GW ~= 666,667 H100-class chips. OpenAI's 2.00M H100-eq
  ~= 1.54M physical accelerators (GB200 ~3x/chip) ~= 2.3 GW across ALL clouds
  (Azure + Oracle + CoreWeave + Stargate); Stargate/Abilene alone ~0.3 GW l

**Read.** OpenAI runs ~**2.0M H100-equivalents**, Anthropic ~**1.7M** (range 1.0–3.0M) — performance-normalized, not physical chip counts (a GB200 is ~3× an H100 per chip). OpenAI's split is ~50/50 (inference rising from Epoch's 44% mid-2025); Anthropic is inference-tilted (~65%) because it offloads serving to Google TPU and leased NVIDIA while AWS Trainium anchors training. Industry-wide, inference is already the majority of compute.

## Slide 2 — Atlas Cloud margin on DeepSeek V4 Pro

**Cost basis:** one 8×NVIDIA H200 node serving the 1.6T-param / 49B-active MoE in FP4 (the ~862 GB checkpoint fits a single 1,128 GB node).

**Key physics:** input tokens = **prefill** (compute-bound, parallel, cheap, high throughput); output tokens = **decode** (memory-bandwidth-bound, one token at a time, expensive). That asymmetry is why every provider prices output ~2× input — even though output costs ~5× input to serve.

We bundle assumptions into three scenarios (efficient / base / conservative) and stress-test the verdict.

In [3]:
ATLAS_PRICE_IN, ATLAS_PRICE_OUT = 1.68, 3.38      # $/1M tokens (Atlas Cloud published price)
DEEPSEEK_PRICE  = (0.435, 0.87)                   # DeepSeek first-party API = market floor
PREMIUM_CLUSTER = (1.74, 3.48)                    # DeepInfra / Fireworks / Novita premium tier

@dataclass
class Scenario:
    name: str; gpu_hr: float; prefill_tok_s: float; decode_tok_s: float; util: float
    NODE_GPUS = 8
    def node_hr(self):  return self.gpu_hr * self.NODE_GPUS
    def cost_in(self):  return self.node_hr() / (self.prefill_tok_s * 3600 * self.util / 1e6)
    def cost_out(self): return self.node_hr() / (self.decode_tok_s  * 3600 * self.util / 1e6)

SCENARIOS = [
    Scenario('efficient',    1.50, 180_000, 35_000, 0.65),
    Scenario('base',         3.00, 110_000, 22_000, 0.50),
    Scenario('conservative', 6.00,  60_000, 12_000, 0.35),
]

def margin(price, cost): return (price - cost) / price

print(f"Atlas price: ${ATLAS_PRICE_IN}/M in, ${ATLAS_PRICE_OUT}/M out")
print(f"DeepSeek floor ${DEEPSEEK_PRICE[0]}/${DEEPSEEK_PRICE[1]} -> Atlas is {ATLAS_PRICE_IN/DEEPSEEK_PRICE[0]:.1f}x the floor")
print(f"Premium cluster ${PREMIUM_CLUSTER[0]}/${PREMIUM_CLUSTER[1]} -> Atlas undercuts output by {(1-ATLAS_PRICE_OUT/PREMIUM_CLUSTER[1])*100:.0f}%\n")

print(f"{'scenario':<14}{'$/M in':>9}{'$/M out':>9}{'margin in':>11}{'margin out':>12}")
for s in SCENARIOS:
    ci, co = s.cost_in(), s.cost_out()
    print(f"{s.name:<14}{ci:>9.3f}{co:>9.3f}{pct(margin(ATLAS_PRICE_IN, ci)):>11}{pct(margin(ATLAS_PRICE_OUT, co)):>12}")

Atlas price: $1.68/M in, $3.38/M out
DeepSeek floor $0.435/$0.87 -> Atlas is 3.9x the floor
Premium cluster $1.74/$3.48 -> Atlas undercuts output by 3%

scenario         $/M in  $/M out  margin in  margin out
efficient         0.028    0.147        98%         96%
base              0.121    0.606        93%         82%
conservative      0.635    3.175        62%          6%


In [4]:
base = SCENARIOS[1]
ci, co = base.cost_in(), base.cost_out()
print('Base case (8x H200, $3/GPU-hr, 50% utilization):')
print(f'  cost/M input  = ${ci:.3f}  -> margin {pct(margin(ATLAS_PRICE_IN, ci))}')
print(f'  cost/M output = ${co:.3f}  -> margin {pct(margin(ATLAS_PRICE_OUT, co))}')
print(f'  output costs {co/ci:.1f}x input to serve, priced only {ATLAS_PRICE_OUT/ATLAS_PRICE_IN:.1f}x.\n')

def blended(in_share, c_in, c_out):
    out_share = 1 - in_share
    rev  = ATLAS_PRICE_IN*in_share + ATLAS_PRICE_OUT*out_share
    cost = c_in*in_share + c_out*out_share
    return rev, cost, (rev-cost)/rev

MIXES = [(0.8,'input-heavy 4:1'), (0.5,'balanced 1:1'), (0.25,'output-heavy 1:3 (reasoning)')]
print('Blended gross margin by traffic mix (base case):')
for s, label in MIXES:
    rev, cost, m = blended(s, ci, co)
    print(f'  {label:<30} margin {pct(m)}  (rev ${rev:.2f} vs cost ${cost:.2f})')

cons = SCENARIOS[2]
ci_c, co_c = cons.cost_in(), cons.cost_out()
print('\nSame mixes under the pessimistic cost basis (stress test, single-node penalty):')
for s, label in MIXES:
    rev, cost, m = blended(s, ci_c, co_c)
    print(f'  {label:<30} margin {pct(m)}  (rev ${rev:.2f} vs cost ${cost:.2f})')

print('\nVerdict: positive gross margin on compute in EVERY scenario. Base case ~80-90%;')
print(f'in the pessimistic case blended margin stays positive but output alone narrows to {pct(margin(ATLAS_PRICE_OUT, co_c))}')
print('(decode is the binding constraint).')
print('Caveat: GROSS margin on marginal compute only — excludes idle capacity,')
print('free tier, networking/egress, support and R&D.')


Base case (8x H200, $3/GPU-hr, 50% utilization):
  cost/M input  = $0.121  -> margin 93%
  cost/M output = $0.606  -> margin 82%
  output costs 5.0x input to serve, priced only 2.0x.

Blended gross margin by traffic mix (base case):
  input-heavy 4:1                margin 89%  (rev $2.02 vs cost $0.22)
  balanced 1:1                   margin 86%  (rev $2.53 vs cost $0.36)
  output-heavy 1:3 (reasoning)   margin 84%  (rev $2.96 vs cost $0.48)

Same mixes under the pessimistic cost basis (stress test, single-node penalty):
  input-heavy 4:1                margin 43%  (rev $2.02 vs cost $1.14)
  balanced 1:1                   margin 25%  (rev $2.53 vs cost $1.90)
  output-heavy 1:3 (reasoning)   margin 14%  (rev $2.96 vs cost $2.54)

Verdict: positive gross margin on compute in EVERY scenario. Base case ~80-90%;
in the pessimistic case blended margin stays positive but output alone narrows to 6%
(decode is the binding constraint).
Caveat: GROSS margin on marginal compute only — excludes i

## Slide 2 — Bottom-up TCO: node cost from first principles

The base case above uses a **market GPU rental rate** ($3/GPU-hr) as the cost basis. That rate is already *fully loaded*: it embeds datacenter, power, cooling, networking, hardware amortization **and the lessor's margin**.

Here we instead **own** the hardware and add every line item explicitly. Two things fall out: (a) **hardware amortization dominates (~76%)** while power+cooling is only ~5%, and (b) the owned-TCO cost lands **at or below** the rental proxy — so the rental-based margin is, if anything, conservative. Assumptions: see [`sources.md`](sources.md) → *Bottom-up TCO*.

In [5]:
@dataclass
class NodeTCO:
    """All-in $/wall-clock-hour of OWNING and OPERATING one 8xH200 node."""
    name: str; gpu_price: float; node_overhead_capex: float; life_years: float
    node_kw_it: float; pue: float; elec_usd_kwh: float; datacenter_hr: float; staff_hr: float
    GPUS = 8; HOURS_PER_YEAR = 8760
    def node_capex(self):  return self.GPUS*self.gpu_price + self.node_overhead_capex
    def hardware_hr(self): return self.node_capex() / (self.life_years*self.HOURS_PER_YEAR)
    def power_hr(self):    return self.node_kw_it * self.pue * self.elec_usd_kwh
    def node_hr(self):     return self.hardware_hr()+self.power_hr()+self.datacenter_hr+self.staff_hr
    def breakdown(self):
        return {'hardware amort':self.hardware_hr(),'power+cooling':self.power_hr(),
                'datacenter+net':self.datacenter_hr,'staff+ops':self.staff_hr}

TCO = [
    NodeTCO('TCO-low',  28_000, 88_000,  5, 10, 1.20, 0.05, 1.0, 0.75),
    NodeTCO('TCO-base', 32_000, 120_000, 3, 10, 1.25, 0.08, 2.0, 1.50),
    NodeTCO('TCO-high', 40_000, 120_000, 3, 10, 1.30, 0.12, 3.5, 2.50),
]

def cost_per_mtok(node_hr, tok_s, util):  return node_hr / (tok_s*3600*util/1e6)

prefill, decode, util = base.prefill_tok_s, base.decode_tok_s, base.util
print(f"{'TCO line ($/hr)':<18}{'low':>9}{'base':>9}{'high':>9}")
for k in TCO[1].breakdown():
    r=[t.breakdown()[k] for t in TCO]; print(f'{k:<18}{r[0]:>9.2f}{r[1]:>9.2f}{r[2]:>9.2f}')
tot=[t.node_hr() for t in TCO]
print(f"{'NODE TOTAL $/hr':<18}{tot[0]:>9.2f}{tot[1]:>9.2f}{tot[2]:>9.2f}")
print(f"{'= $/GPU-hr':<18}{tot[0]/8:>9.2f}{tot[1]/8:>9.2f}{tot[2]/8:>9.2f}")
print(f"\nComposition (base): hardware {pct(TCO[1].hardware_hr()/TCO[1].node_hr())}, power+cooling {pct(TCO[1].power_hr()/TCO[1].node_hr())}.")

print('\nCost per 1M tokens by basis (same throughput, util 50%):')
print(f"{'basis':<22}{'node $/hr':>11}{'$/M in':>9}{'$/M out':>9}{'margin in':>11}{'margin out':>12}")
for label,nh in [('rental proxy ($3/GPU)', base.node_hr())]+[(t.name,t.node_hr()) for t in TCO]:
    ci=cost_per_mtok(nh,prefill,util); co=cost_per_mtok(nh,decode,util)
    print(f'{label:<22}{nh:>11.2f}{ci:>9.3f}{co:>9.3f}{pct(margin(ATLAS_PRICE_IN,ci)):>11}{pct(margin(ATLAS_PRICE_OUT,co)):>12}')

print(f"\nVerdict: owned TCO base (~${TCO[1].node_hr():.0f}/hr) <= rental proxy (${base.node_hr():.0f}/hr) -> margin is conservative.")
print('Outside this COGS view: idle/unsold capacity, free tier, egress, sales & marketing, support, R&D.')

TCO line ($/hr)         low     base     high
hardware amort         7.12    14.31    16.74
power+cooling          0.60     1.00     1.56
datacenter+net         1.00     2.00     3.50
staff+ops              0.75     1.50     2.50
NODE TOTAL $/hr        9.47    18.81    24.30
= $/GPU-hr             1.18     2.35     3.04

Composition (base): hardware 76%, power+cooling 5%.

Cost per 1M tokens by basis (same throughput, util 50%):
basis                   node $/hr   $/M in  $/M out  margin in  margin out
rental proxy ($3/GPU)       24.00    0.121    0.606        93%         82%
TCO-low                      9.47    0.048    0.239        97%         93%
TCO-base                    18.81    0.095    0.475        94%         86%
TCO-high                    24.30    0.123    0.614        93%         82%

Verdict: owned TCO base (~$19/hr) <= rental proxy ($24/hr) -> margin is conservative.
Outside this COGS view: idle/unsold capacity, free tier, egress, sales & marketing, support, R&D.
